In [ ]:
from ase.io import Trajectory

In [7]:
def load_first_50ps_frames(traj_path):
    """
    Load the first 5000 frames (assumed 50 ps) from a trajectory file.
    """
    with Trajectory(traj_path) as traj_file:
        return [traj_file[i] for i in range(5000)]

# 323K data
traj_path = "/global/homes/y/yuejian/project/MLFF-distill/m4558/323K_500ps_trajs/md_omol_naotf_diglyme_1m_s1p1/md_omol_naotf_diglyme_1m_s1p1.traj"
first_50ps_frames_323K = load_first_50ps_frames(traj_path)

# 293K data
traj_path = "/global/homes/y/yuejian/project/MLFF-distill/m5024/UMA_trajs_293K/naotf_diglyme.traj"
first_50ps_frames_293K = load_first_50ps_frames(traj_path)

In [9]:
import numpy as np
import matplotlib.pyplot as plt
# from ase.geometry import supercell
from ase.geometry import analysis
from ase.geometry import get_distances
from ase import Atoms
from scipy.stats import ks_2samp, wasserstein_distance, iqr, entropy
from scipy.spatial.distance import pdist, squareform

def select_heavy_atoms(atoms):
    """Return indices of heavy atoms (atomic number > 1)."""
    numbers = atoms.get_atomic_numbers()
    return np.where(numbers > 1)[0]

def get_heavy_positions_list(traj):
    """For a trajectory (list of Atoms), get heavy atom coordinates as (n_frames, n_heavy, 3) array and check indices."""
    idx = select_heavy_atoms(traj[0])
    symbols0 = traj[0][idx].get_chemical_symbols()
    arr = np.empty((len(traj), len(idx), 3))
    for f, at in enumerate(traj):
        idx_now = select_heavy_atoms(at)
        if not np.array_equal(idx, idx_now):
            raise ValueError(f"Heavy atom indices differ at frame {f}: {idx} vs {idx_now}")
        if at[idx].get_chemical_symbols() != symbols0:
            raise ValueError(f"Heavy atom order/symbols differ at frame {f}")
        arr[f] = at.get_positions()[idx]
    return arr, idx, symbols0

def kabsch_align(P, Q):
    """
    Align P (Nx3) onto Q (Nx3) using the Kabsch algorithm (modifies P).
    Returns rotated P.
    """
    # Center
    Pc = P - P.mean(axis=0)
    Qc = Q - Q.mean(axis=0)
    # Covariance
    C = Pc.T @ Qc
    V, S, Wt = np.linalg.svd(C)
    d = np.sign(np.linalg.det(V @ Wt))
    U = V @ np.diag([1,1,d]) @ Wt
    P_rot = Pc @ U
    return P_rot, Qc

def rmsd(P, Q):
    return np.sqrt(np.mean(np.sum((P - Q)**2, axis=1)))

def get_rmsd_vs_first(heavy_coords_arr):
    """
    Returns 1D array of RMSD from each frame (aligned) to the first frame.
    """
    ref = heavy_coords_arr[0]
    out = np.zeros(len(heavy_coords_arr))
    for i, xyz in enumerate(heavy_coords_arr):
        xyz_aln, ref_aln = kabsch_align(xyz, ref)
        out[i] = rmsd(xyz_aln, ref_aln)
    return out

def subsample_frames(heavy_coords_arr, stride=10):
    """Subsample frames as (n_sub, n_heavy, 3). Returns array and indices."""
    idxs = np.arange(0, heavy_coords_arr.shape[0], stride)
    return heavy_coords_arr[idxs], idxs

def pairwise_rmsd_matrix(coords):  # coords: (n_frames, n_atoms, 3)
    """Returns condensed distance matrix (n*n-1/2) (upper triangle), using Kabsch alignment to minimum."""
    n = coords.shape[0]
    pdmat = np.zeros((n, n))
    for i in range(n):
        for j in range(i+1, n):
            a, b = coords[i], coords[j]
            a_aln, b_aln = kabsch_align(a, b)
            r = rmsd(a_aln, b_aln)
            pdmat[i, j] = r
    return squareform(pdmat)  # (n*(n-1)/2,)

def compute_summary_stats(data):
    data = np.asarray(data)
    return {
        "mean": np.mean(data),
        "median": np.median(data),
        "std": np.std(data),
        "iqr": iqr(data),
        "90th_perc": np.percentile(data, 90),
        "95th_perc": np.percentile(data, 95)
    }

def jensen_shannon_divergence(p, q):
    # p, q: normalized histograms
    p = np.asarray(p) + 1e-12
    q = np.asarray(q) + 1e-12
    p /= p.sum()
    q /= q.sum()
    m = 0.5*(p+q)
    return 0.5 * (entropy(p, m) + entropy(q, m))

### --- Core analysis ---

# 1) Heavy atom checks and coordinates
coords_323K, idx_323K, syms_323K = get_heavy_positions_list(first_50ps_frames_323K)
coords_293K, idx_293K, syms_293K = get_heavy_positions_list(first_50ps_frames_293K)
if len(idx_323K) != len(idx_293K):
    raise ValueError(f"Number of heavy atoms differs: 323K={len(idx_323K)}, 293K={len(idx_293K)}")
if syms_323K != syms_293K:
    raise ValueError(f"Heavy atom order or elements differ between trajectories.")

# 2) RMSD vs first-frame (50ps, dt=10fs)
dt_fs = 10
times_ps = np.arange(coords_323K.shape[0]) * dt_fs * 1e-3  # ps

rmsd323 = get_rmsd_vs_first(coords_323K)
rmsd293 = get_rmsd_vs_first(coords_293K)

# 3) Pairwise RMSD (diversity metric)
stride = 10  # configurable: set to 1 for dense, 10 for 500 frames from 5000
coords_sub_323K, idxs_323K = subsample_frames(coords_323K, stride=stride)
coords_sub_293K, idxs_293K = subsample_frames(coords_293K, stride=stride)
pairw_323K = pairwise_rmsd_matrix(coords_sub_323K)
pairw_293K = pairwise_rmsd_matrix(coords_sub_293K)
stats_323 = compute_summary_stats(pairw_323K)
stats_293 = compute_summary_stats(pairw_293K)

# 4) Compare distributions

# KS and Wasserstein between RMSD-to-first distributions
ks_stat, ks_p = ks_2samp(rmsd323, rmsd293)
try:
    wass_dist = wasserstein_distance(rmsd323, rmsd293)
except Exception:
    wass_dist = np.nan

# Jensen-Shannon: compute histograms
n_bins = 60
bins = np.linspace(0, max(rmsd323.max(), rmsd293.max()), n_bins+1)
h323, _ = np.histogram(rmsd323, bins=bins)
h293, _ = np.histogram(rmsd293, bins=bins)
jsd = jensen_shannon_divergence(h323, h293)

# 5) Plotting

plt.figure(figsize=(7,4))
plt.plot(times_ps, rmsd323, label="323 K", alpha=0.85)
plt.plot(times_ps, rmsd293, label="293 K", alpha=0.85)
plt.xlabel("Time (ps)")
plt.ylabel("RMSD to first frame (Å)")
plt.title("RMSD vs Time (Heavy Atoms, Aligned)")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(6,4))
bin_centers = 0.5*(bins[1:]+bins[:-1])
plt.step(bin_centers, h323/np.sum(h323), where='mid', label="323 K")
plt.step(bin_centers, h293/np.sum(h293), where='mid', label="293 K")
plt.xlabel("RMSD to first frame (Å)")
plt.ylabel("Probability density")
plt.title("RMSD Distribution (Heavy Atoms)")
plt.legend()
plt.tight_layout()
plt.show()

# Pairwise RMSD "diversity" hist
pw_bins = np.linspace(0, max(pairw_323K.max(), pairw_293K.max(), 2.0), n_bins)
plt.figure(figsize=(6,4))
plt.hist(pairw_323K, bins=pw_bins, alpha=0.5, label="323 K", density=True, histtype="stepfilled")
plt.hist(pairw_293K, bins=pw_bins, alpha=0.5, label="293 K", density=True, histtype="stepfilled")
plt.xlabel("Pairwise RMSD (Å)")
plt.ylabel("Probability density")
plt.title(f"Pairwise RMSD 'Diversity' (Stride={stride})")
plt.legend()
plt.tight_layout()
plt.show()

# 6) Text summary

print("\n--- RMSD vs First Frame: Distribution Comparison ---")
print(f"323 K: mean={np.mean(rmsd323):.3f}, std={np.std(rmsd323):.3f}, median={np.median(rmsd323):.3f}, max={np.max(rmsd323):.3f}")
print(f"293 K: mean={np.mean(rmsd293):.3f}, std={np.std(rmsd293):.3f}, median={np.median(rmsd293):.3f}, max={np.max(rmsd293):.3f}")
print(f"KS statistic={ks_stat:.4f}, p-value={ks_p:.3e}")
print(f"Jensen-Shannon divergence={jsd:.4f}")
if not np.isnan(wass_dist):
    print(f"Wasserstein distance={wass_dist:.4f}\n")

print("--- Pairwise RMSD Distribution (Diversity, Heavy Atoms, stride=%d) ---" % stride)
print("  323K:", ", ".join(f"{k}={v:.3f}" for k,v in stats_323.items()))
print("  293K:", ", ".join(f"{k}={v:.3f}" for k,v in stats_293.items()))

if stats_323["mean"] > stats_293["mean"]:
    more = "greater"
elif stats_323["mean"] < stats_293["mean"]:
    more = "smaller"
else:
    more = "equal"
print(f"\nSUMMARY: At 323K, conformational diversity ({stats_323['mean']:.3f} Å mean pairwise RMSD) is {more} than at 293K ({stats_293['mean']:.3f} Å).")

print("\nNOTES:")
print(f"  Used only heavy atoms (n={len(idx_323K)}), same order/chemistry checked for both trajectories.")
print(f"  RMSD computed after optimal superposition (no PBC unwrapping performed).")
print(f"  Frame stride for diversity: {stride}. Adjust in code if desired.")

ValueError: Distance matrix 'X' must be symmetric.